In [29]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

from langchain_community.vectorstores import FAISS


In [20]:
load_dotenv()

True

In [23]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

DATA_FILE_PATH = Path("data.txt")

In [24]:
print(DATA_FILE_PATH)

data.txt


In [25]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'data.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probation, employe

In [26]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [12]:
print(documents[0].metadata)

{'source': 'data.txt'}


In [13]:
print(len(documents[0].page_content))

2597


In [42]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000,
  chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.'), Document(metadata={'source': 'data.txt'}, page_content='3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 month

In [43]:
embeddings_model  = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

vector_store = FAISS.from_documents(chunks, embeddings_model)

In [44]:
test_query = "how many sick leaves do employees get"

top_matches = vector_store.similarity_search(test_query, k=2)

print(test_query)

for i,match in enumerate(top_matches, start=1):
    print(f"Match {i}:")
    print(match.page_content)

how many sick leaves do employees get
Match 1:
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.
Match 2:
6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workpl

## tool

In [40]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str):
    """
    search the HR policy document for information about leave, work from home, probation, notice period, reimbursement, code of conduct, holidays or exit process
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

## DATA RETREIVAL


In [36]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [38]:
from langchain.agents import create_agent

In [41]:
hr_assistant = create_agent(
    model = llm,
    tools =[search_hr_policy],
    system_prompt = """
    You are a friendly HR assistant,
    Always use the search_hr_policy tool to look up the facts before answering.
    If the answer isn't in the search results, say you dont know instead of guessing.
    """
)

In [54]:
def ask_hr_assistant(question:str):
    """send a question to the RAG agent and print a nicely formatted answer"""
    print("="*60)
    print(f"Question: {question}")
    print("-"*60)

    response = hr_assistant.invoke(
      {
        "messages":[
            {
                "role":"user",
                "content": question
            }
        ]
      })
    answer = response["messages"][-1].content

    print("ANSWER", answer)
    print("="*60)
    print()
    return answer

In [55]:
ask_hr_assistant("how many sick leaves do employees get")

Question: how many sick leaves do employees get
------------------------------------------------------------
ANSWER Employees are entitled to **10 paid sick days per year**. A medical certificate is required if the sick leave extends beyond two consecutive days.



'Employees are entitled to **10 paid sick days per year**. A medical certificate is required if the sick leave extends beyond two consecutive days.'

In [50]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "how many sick leaves do employees get"
            }
        ]
    }
)

In [51]:
response["messages"][-1].content

'Employees are entitled to **10 paid sick days per year**. A medical certificate is required if the sick leave extends beyond two consecutive days.'